In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, box
import matplotlib.pyplot as plt
import pyexcel_ods
import openpyxl


# GEM files
# gem_solar_under_path = "GEM/Global-Solar-Power-Tracker-June-2024_under20MW.csv"
# gem_solar_over_path = "GEM/Global-Solar-Power-Tracker-June-2024_over20MW.csv"
# gem_wind_under_path = "GEM/Global-Wind-Power-Tracker-June-2024_under10MW.csv"
# gem_wind_over_path = "GEM/Global-Wind-Power-Tracker-June-2024_over10MW.csv"
# gem_nuclear_path = "GEM/Global-Nuclear-Power-Tracker-October-2023.csv"

# gem_solar_under_path = "GEM/Global-Solar-Power-Tracker-February-2025_under20MW.csv"
# gem_solar_over_path = "GEM/Global-Solar-Power-Tracker-February-2025_over20MW.csv"
# gem_wind_under_path = "GEM/Global-Wind-Power-Tracker-February-2025_under10MW.csv"
# gem_wind_over_path = "GEM/Global-Wind-Power-Tracker-February-2025_over10MW.csv"
# gem_nuclear_path = "GEM/Global-Nuclear-Power-Tracker-July-2024.csv"

excel_solar = "GEM/raw_data/Global-Solar-Power-Tracker-February-2025.xlsx"
excel_wind = "GEM/raw_data/Global-Wind-Power-Tracker-February-2025.xlsx"
excel_nuclear = "GEM/raw_data/Global-Nuclear-Power-Tracker-July-2024.xlsx"

# GEM output file
gem_output_path = "gem_installed_capacities.csv"

# year for estimation
year = 2050

# Techno-economic file
ods_file = "highres_technoeconomic_database.ods"

existing_pcap_r_path = f"existing_pcap_r_{year}.csv"
existing_pcap_z_path = f"existing_pcap_z_{year}.csv"
gen_lim_z_path = f"gen_lim_z_{year}.csv"

output_file_path = "highres_technoeconomic_database_updated.ods"


# Technologies used in GEM
pv_types = [
    "PV",
    "Assumed PV"
]
Offshore_types = [
    "Offshore mount unknown",
    "Offshore hard mount",
    "Offshore floating"
]


# shape files
onshore_shapefile = "europe_onshore.geojson"
offshore_shapefile = "europe_offshore.geojson"

In [3]:
# open ODS (Excel) file

excel_exist_pcap_z = pd.read_excel(
    ods_file,
    sheet_name="gen_exist_z",
    skiprows=0,
    engine="calamine",
)
excel_exist_pcap_r = pd.read_excel(
    ods_file,
    sheet_name="gen_exist_r",
    skiprows=0,
    engine="calamine",
)

excel_gen_lim_z = pd.read_excel(
    ods_file,
    sheet_name="gen_lim_z",
    skiprows=0,
    engine="calamine",
)

zones = [col for col in excel_gen_lim_z.columns if col not in ["Technology","Year","parameter","limtype"]]

In [4]:
country_codes = {
    "Albania" : "AL",
    "Austria" : "AT",
    "Bosnia and Herzegovina": "BA",
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Switzerland" : "CH",
    "Cyprus" : "CY",
    "Czech Republic" : "CZ",
    "Germany" : "DE",
    "Denmark" : "DK",
    "Estonia" : "EE" ,
    "Spain" : "ES",
    "Finland" : "FI",
    "France" : "FR",
    "United Kingdom" : "UK",
    "Greece" : "GR",
    "Croatia" : "HR",
    "Hungary" : "HU",
    "Ireland" : "IE",
    "Iceland" : "IS",
    "Italy" : "IT",
    "Liechtenstein" : "LI",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Latvia" : "LV",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Norway" : "NO",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Serbia" : "RS",
    "Sweden" : "SE",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Kosovo" : "XK",
}

In [5]:

### Solar
# gem_solar_over = pd.read_csv(gem_solar_over_path,sep=';')
# gem_solar_under = pd.read_csv(gem_solar_under_path,sep=';')
# gem_solar = pd.concat([gem_solar_over, gem_solar_under])

gem_solar = pd.concat(pd.read_excel(
    excel_solar,
    sheet_name=["20 MW+","1-20 MW"],
    skiprows=0,
))
gem_solar = (
    gem_solar
    .rename(columns={'Project Name' : 'project_name','Country/Area' : 'country', 
                     'Technology Type' :'Technology','Capacity (MW)' : 'capacity_MW',
                     'Status':'status','Latitude':'lat','Longitude':'lon',
                     'Start year':'commissioning_year','Retired year':'retirement_year'})
    .query('Technology == @pv_types')
    .query('lat.notna() & lon.notna()')
    .drop_duplicates(subset=['project_name','capacity_MW'])
    .loc[:,['project_name','country','capacity_MW','status','lat','lon','commissioning_year','retirement_year']]
)
### Wind"europe_offshore.geojson",
# gem_wind_over10MW = pd.read_csv(gem_wind_over_path,sep=';')
# gem_wind_under10MW = pd.read_csv(gem_wind_under_path,sep=';')
# gem_wind = pd.concat([gem_wind_over10MW, gem_wind_under10MW])

gem_wind = pd.concat(pd.read_excel(
    excel_wind,
    sheet_name=["Data","Below Threshold"],
    skiprows=0,
))
gem_wind = (
    gem_wind
    .rename(columns={'Project Name' : 'project_name','Country/Area' : 'country', 
                     'Installation Type' : 'Technology','Capacity (MW)' : 'capacity_MW',
                     'Status':'status','Latitude':'lat','Longitude':'lon',
                     'Start year':'commissioning_year','Retired year':'retirement_year'})
    .query('lat.notna() & lon.notna()')
    .drop_duplicates(subset=['project_name','capacity_MW'])
    .loc[:,['project_name','country','capacity_MW','Technology','status','lat','lon','commissioning_year','retirement_year']]
)

gem_onshore = (
    gem_wind
    .query("Technology == 'Onshore'")
    .drop('Technology',axis=1)
    .drop_duplicates(subset=['project_name','capacity_MW'])
)

gem_offshore = (
    gem_wind
    .query("Technology == @Offshore_types")
    .drop('Technology',axis=1)
    .drop_duplicates(subset=['project_name','capacity_MW'])
)
### Nuclear
# gem_nuclear = pd.read_csv(gem_nuclear_path, sep=';', decimal=",")
gem_nuclear = pd.concat(pd.read_excel(
    excel_nuclear,
    sheet_name=["Data"],
    skiprows=0,
))

gem_nuclear = (
    gem_nuclear
    .rename(columns={'Project Name' : 'project_name','Start Year' : 'commissioning_year', 'Construction Start Date':'construction_start',
                     'Retirement Year' : 'retirement_year', 'Cancellation Year':'cancellation_year',
                     'Capacity (MW)' : 'capacity_MW','Country/Area':'country','Status':'status', 'Latitude':'lat','Longitude':'lon'})
    .loc[:,['country','capacity_MW','project_name','status','lat','lon','commissioning_year','construction_start','retirement_year', 'cancellation_year']]
)


In [6]:
# Merge onshore, offshore, solar, and nuclear
merged_gem = (
    pd.concat([
        (
            gem_onshore
            .assign(Technology = "onshore")
        ),
        (
            gem_offshore
            .assign(Technology = "offshore")
        ),
        (
            gem_solar
            .assign(Technology = "solar")
        ),
        (
            gem_nuclear
            # These columns are not relevant
            .drop(columns=["construction_start","cancellation_year"])
            .assign(Technology = "nuclear")
        ),
    ])
)

merged_gem = (
    merged_gem
    # Add country code
    .assign(iso_code = merged_gem["country"].replace(country_codes))
    # Only select the relevant countries
    .query("iso_code == @zones")
)


In [7]:
# Calculate lifetime of Solar, Onshore, Offshore and Nuclear based on GEM data
# with retirement year and commissiong year

lifetime = (
    merged_gem
    .query('commissioning_year.notnull() & retirement_year.notnull()')
    .assign(lifetime_year = lambda x : x.retirement_year - x.commissioning_year)
    .loc[:,["country","Technology","lifetime_year"]]
)

# lifetime.groupby(["country","Technology"]).describe()
lifetime.groupby(["Technology"]).describe()



lifetime_year                                                      
                   count       mean        std   min    25%   50%    75%   max
Technology                                                                    
nuclear            124.0  30.354839  14.451598   0.0  21.00  31.5  41.00  73.0
offshore             2.0  16.500000   2.121320  15.0  15.75  16.5  17.25  18.0
onshore            112.0  15.928571   5.942236   1.0  14.00  17.0  20.00  25.0
solar               59.0  20.779661   7.593254   0.0  20.00  25.0  25.00  25.0

In [8]:
# Only plants with status "operating"
filtered_gem = (
    merged_gem
    .query("status == ['operating']")
    .loc[:,["project_name","Technology","country","iso_code","lat","lon","capacity_MW","commissioning_year"]]
    .reset_index(drop=True)
)
filtered_gem

,project_name,Technology,country,iso_code,lat,lon,capacity_MW,commissioning_year
0,Albrechtsfeld wind farm,onshore,Austria,AT,47.83020,17.0197,30.0,2013.0
1,Andau wind farm,onshore,Austria,AT,47.82380,17.0494,33.0,2013.0
2,Andau wind farm,onshore,Austria,AT,47.79640,17.0391,81.0,2014.0
3,Andlersdorf/Orth wind farm,onshore,Austria,AT,48.18380,16.6864,39.0,2017.0
4,Au Am Leithaberge wind farm,onshore,Austria,AT,47.93700,16.5345,18.0,2018.0
...,...,...,...,...,...,...,...,...
23880,Heysham nuclear power plant,nuclear,United Kingdom,UK,54.03101,-2.9112,680.0,1989.0
23881,Heysham nuclear power plant,nuclear,United Kingdom,UK,54.03101,-2.9112,680.0,1989.0
23882,Sizewell nuclear power plant,nuclear,United Kingdom,UK,52.21450,1.6206,1250.0,1995.0
23883,Torness nuclear power plant,nuclear,United Kingdom,UK,55.96790,-2.4086,682.0,1988.0


In [9]:
# Estimate decomissionned year based on historical data (lifeftime) by technology

lifetime_tech_dict = {
    "nuclear": (
        lifetime
        .query("Technology=='nuclear'")
        .loc[:,"lifetime_year"]
        .quantile(0.75)
    ),
    "onshore": 30,
    "offshore": 30,
    "solar": 30,        
}

# Assign lifetime to every technology
gem_dataset = filtered_gem.copy()
for tech, lifetime_value in lifetime_tech_dict.items():
    gem_dataset.loc[gem_dataset["Technology"] == tech, "estimated_retirement_year"] = (
        gem_dataset.loc[gem_dataset["Technology"] == tech, "commissioning_year"] + lifetime_value
    )
# Fix format issue and nan values (no commissioning year available)
gem_dataset["commissioning_year"] = (
    gem_dataset["commissioning_year"]
    .fillna(0)
    .round(0)
    .astype(int)
)
gem_dataset["estimated_retirement_year"] = (
    gem_dataset["estimated_retirement_year"]
    .fillna(0)
    .round(0)
    .astype(int)
)


gem_dataset

,project_name,Technology,country,iso_code,lat,lon,capacity_MW,commissioning_year,estimated_retirement_year
0,Albrechtsfeld wind farm,onshore,Austria,AT,47.83020,17.0197,30.0,2013,2043
1,Andau wind farm,onshore,Austria,AT,47.82380,17.0494,33.0,2013,2043
2,Andau wind farm,onshore,Austria,AT,47.79640,17.0391,81.0,2014,2044
3,Andlersdorf/Orth wind farm,onshore,Austria,AT,48.18380,16.6864,39.0,2017,2047
4,Au Am Leithaberge wind farm,onshore,Austria,AT,47.93700,16.5345,18.0,2018,2048
...,...,...,...,...,...,...,...,...,...
23880,Heysham nuclear power plant,nuclear,United Kingdom,UK,54.03101,-2.9112,680.0,1989,2030
23881,Heysham nuclear power plant,nuclear,United Kingdom,UK,54.03101,-2.9112,680.0,1989,2030
23882,Sizewell nuclear power plant,nuclear,United Kingdom,UK,52.21450,1.6206,1250.0,1995,2036
23883,Torness nuclear power plant,nuclear,United Kingdom,UK,55.96790,-2.4086,682.0,1988,2029


In [10]:
# Save dataset
gem_dataset.to_csv(gem_output_path, index=False)

# Updating techno-economic file

In [11]:
# get shapefiles from intermediate data from highres WF
geodata_files = {
    "onshore": onshore_shapefile,
    "offshore_bottom": offshore_shapefile,
}

# Load onshore shapefile
euroshape = gpd.read_file(geodata_files["onshore"]).set_index(["index"])
aggr_countries = euroshape.CNTR_CODE.unique().tolist() # get relevant countries
aggr_countries

['ES',
 'PT',
 'FR',
 'IT',
 'CH',
 'DE',
 'BE',
 'LU',
 'UK',
 'IE',
 'NL',
 'GR',
 'HR',
 'SI',
 'BG',
 'RO',
 'HU',
 'SK',
 'AT',
 'CZ',
 'PL',
 'DK',
 'LT',
 'LV',
 'SE',
 'EE',
 'FI',
 'NO']

In [12]:
# Calculate boundaries based on onshore and offshore shapefiles
boundaries = []
for geodata_file_name, geodata_file_path in geodata_files.items():
    boundaries.append(gpd.read_file(geodata_file_path).set_index(["index"]))

boundaries = pd.concat(boundaries).bounds

boundaries = boundaries.groupby(lambda x: "bountry").agg(
    {"minx": "min", "miny": "min", "maxx": "max", "maxy": "max"}
)

boundaries

,minx,miny,maxx,maxy
index,,,,
bountry,-11.586116,34.9554,31.5218,71.1782


In [13]:
# convert to geopandas
existing_pcap = (
    gpd.GeoDataFrame(
        gem_dataset,
        geometry=gpd.points_from_xy(gem_dataset.lon, gem_dataset.lat),
        crs="epsg:4326",
    )
    .loc[
        :,
        [
            "Technology",
            "iso_code",
            "capacity_MW",
            "geometry",
            "estimated_retirement_year",
        ],
    ]
    .query("iso_code==@aggr_countries") # select only relevant countries
)
existing_pcap

,Technology,iso_code,capacity_MW,geometry,estimated_retirement_year
0,onshore,AT,30.0,POINT (17.0197 47.8302),2043
1,onshore,AT,33.0,POINT (17.0494 47.8238),2043
2,onshore,AT,81.0,POINT (17.0391 47.7964),2044
3,onshore,AT,39.0,POINT (16.6864 48.1838),2047
4,onshore,AT,18.0,POINT (16.5345 47.937),2048
...,...,...,...,...,...
23880,nuclear,UK,680.0,POINT (-2.9112 54.03101),2030
23881,nuclear,UK,680.0,POINT (-2.9112 54.03101),2030
23882,nuclear,UK,1250.0,POINT (1.6206 52.2145),2036
23883,nuclear,UK,682.0,POINT (-2.4086 55.9679),2029


In [14]:
# Clip/Remove plants outside the considered area

# Shapefile boundaries
# boundaries = euroshape.total_bounds
# rectx1 = boundaries[0] # -10.2193 # -11.586115518116628
# rectx2 = boundaries[2] # 31.5218 # 30.856168623982853
# recty1 = boundaries[1] # 34.9554 # 35.12821044935282
# recty2 = boundaries[3] # 71.1782 # 71.03514811479374

rectx1 = boundaries.loc["bountry", "minx"]
rectx2 = boundaries.loc["bountry", "maxx"]
recty1 = boundaries.loc["bountry", "miny"]
recty2 = boundaries.loc["bountry", "maxy"]


polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)
existing_pcap = gpd.clip(existing_pcap, polygon)

In [15]:
# Determine the nuts2 region for every power plant
all_techs = gpd.sjoin(euroshape, existing_pcap, how="right", predicate="contains").loc[
    :,
    [
        "Technology",
        "index",
        "CNTR_CODE",
        "iso_code",
        "capacity_MW",
        "geometry",
        "estimated_retirement_year",
    ],
]

# Matched plants
no_error_techs = (
    all_techs.query("CNTR_CODE == iso_code")  # country codes match
    # conserve "geometry" for plotting (it could be removed)
    .loc[
        :,
        [
            "Technology",
            "iso_code",
            "index",
            "capacity_MW",
            "estimated_retirement_year",
            "geometry",
        ],
    ]
    # .loc[:, ["Technology", "iso_code", "index", "capacity_MW","geometry"]]
)

# Mismatched plants (country codes mismatch)
error_techs = all_techs.query("CNTR_CODE != iso_code")


# Plants with different country assignment or nan assignment
# from shapefile and database
# There are two groups:
# 1) Plants on the coast and offshore
# 2) Plants on the border of two countries


## Plants on the coast and offshore
# Solution: assign closest region"europe_offshore.geojson",
offshore_plants = error_techs.query("CNTR_CODE.isna()").to_crs(3035)  # nan values

for idx, row in offshore_plants.iterrows():
    # Find the closest region
    ztemp = euroshape.loc[euroshape["CNTR_CODE"] == row["iso_code"], :]
    dists = ztemp.to_crs(3035).distance(row.geometry)
    if len(dists) == 0:
        break
    offshore_plants.loc[idx, "index"] = dists.idxmin()

offshore_plants = (
    offshore_plants.to_crs(4326)
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "iso_code",
            "index",
            "capacity_MW",
            "estimated_retirement_year",
            "geometry"
        ],
    ]
)

## Plants on the border of two countries
# Solution: Assign information from shape file
# Issues spotted in Hungary/Austria/Czech Republic
# Only 3 solar plants with 1m resolutiongem_nuclear
country_border_plants = (
    error_techs.query("~CNTR_CODE.isna()")  # different country assignment
)
country_border_plants = (
    country_border_plants
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "CNTR_CODE",
            "index",
            "capacity_MW",
            "estimated_retirement_year",
            "geometry",
        ],
    ].rename(columns={"CNTR_CODE": "iso_code"})
)
"europe_offshore.geojson",
### Merge all the plants
merged_techs = pd.concat(
    [
        no_error_techs,
        offshore_plants,
        country_border_plants,
    ]
).reset_index(drop=True)

In [16]:
# Filtering plants based on the lifetime of the decommissioned plants in GEM

# Nuclear
merged_techs = merged_techs.query(
    "(estimated_retirement_year>=@year)"
).loc[:, ["Technology", "iso_code", "index", "capacity_MW"]]

# Similar idea can be applied for Solar, Onshore, and Offshore

In [17]:
# Format the dataframe like in the ODS (Excel) file

# Technologies with finner resolution
r_techs = ["solar", "onshore", "offshore"]
exist_pcap_r = (
    merged_techs.query("Technology == @r_techs")
    .groupby(["Technology", "index", "iso_code"])
    .sum()
    .reset_index()
    .rename(columns={"capacity_MW": "value", "index": "region", "iso_code": "zone"})
    .assign(parameter="gen_exist_pcap_r")
    .assign(Year=year)
    .assign(limtype="FX") # or FX
    .replace(
        {
            "onshore": "Windonshore",
            "offshore": "Windoffshore",
            "solar": "Solar"
        }
    )
    .reset_index(drop=True)
)

# All capacities aggregated to a coarser resolution
z_techs = ["solar", "onshore", "offshore", "nuclear"]
# z_techs = ["nuclear"]
exist_pcap_z = (
    merged_techs.query("Technology == @z_techs")
    .loc[:, ["Technology", "iso_code", "capacity_MW"]]
    .groupby(["Technology", "iso_code"])
    .sum()
    .reset_index()
    .pivot(index="Technology", columns="iso_code", values="capacity_MW")
    .reset_index("Technology")
    .assign(parameter="gen_exist_pcap_z")
    .assign(Year=year)
    .assign(limtype="FX")
    .replace(  # hardcoded - TODO: get name of technologies
        {
            "onshore": "Windonshore",
            "offshore": "Windoffshore",
            "solar": "Solar",
            "nuclear": "NuclearEPR",
        }
    )
)
exist_pcap_z

iso_code,Technology,AT,BE,BG,CH,DE,DK,EE,ES,FI,...,PL,PT,RO,SE,SI,SK,UK,parameter,Year,limtype
0,NuclearEPR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1720.0,...,NaN,NaN,NaN,NaN,NaN,471.0,NaN,gen_exist_pcap_z,2050,FX
1,Windoffshore,NaN,706.0,NaN,NaN,1452.0,965.6,NaN,NaN,NaN,...,NaN,25.0,NaN,NaN,NaN,NaN,4674.5,gen_exist_pcap_z,2050,FX
2,Windonshore,546.0,417.0,33.6,14.0,7286.0,327.0,429.0,3626.1,5173.0,...,3225.0,266.6,108.0,7447.4,NaN,NaN,1168.1,gen_exist_pcap_z,2050,FX
3,Solar,77.7,39.0,1106.0,1.2,16970.4,1266.8,122.0,12752.1,32.0,...,1628.0,1983.2,399.4,121.0,6.0,NaN,975.1,gen_exist_pcap_z,2050,FX


In [18]:
## Update existing capacities based on gen lim z

zones_exist = [col for col in exist_pcap_z.columns if col not in ["Technology","Year","parameter","limtype"]]
technologies = exist_pcap_z.Technology.unique().tolist()

## Assign nan value to up lim and then fill it
exist_pcap_z_up = exist_pcap_z.copy()
exist_pcap_z_up.loc[:,zones_exist] = np.nan
exist_pcap_z_up["limtype"] = "UP"

for technology in technologies:
    row = excel_gen_lim_z.index[excel_gen_lim_z["Technology"]==technology]
    if len(row)==1:
        for zone in zones_exist:
            col = excel_gen_lim_z.columns[excel_gen_lim_z.columns==zone]
            if len(col)==1:
                gen_lim = excel_gen_lim_z.loc[row[0], col[0]]
                if gen_lim<float(exist_pcap_z.loc[exist_pcap_z.Technology==technology,zone].values[0]):
                    exist_pcap_z_up.loc[exist_pcap_z.Technology==technology,[zone]] = exist_pcap_z.loc[exist_pcap_z.Technology==technology,[zone]]
                    exist_pcap_z.loc[exist_pcap_z.Technology==technology,[zone]] = np.nan

exist_pcap_z_up = exist_pcap_z_up.dropna(subset=zones_exist, how="all")

exist_pcap_z = pd.concat(
    [
        exist_pcap_z,
        exist_pcap_z_up,
    ]
).dropna(subset=zones_exist, how="all")



exist_pcap_z

iso_code,Technology,AT,BE,BG,CH,DE,DK,EE,ES,FI,...,PL,PT,RO,SE,SI,SK,UK,parameter,Year,limtype
0,NuclearEPR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1720.0,...,NaN,NaN,NaN,NaN,NaN,471.0,NaN,gen_exist_pcap_z,2050,FX
1,Windoffshore,NaN,706.0,NaN,NaN,1452.0,965.6,NaN,NaN,NaN,...,NaN,25.0,NaN,NaN,NaN,NaN,4674.5,gen_exist_pcap_z,2050,FX
2,Windonshore,546.0,417.0,33.6,14.0,7286.0,327.0,429.0,3626.1,5173.0,...,3225.0,266.6,108.0,7447.4,NaN,NaN,1168.1,gen_exist_pcap_z,2050,FX
3,Solar,77.7,39.0,1106.0,1.2,16970.4,1266.8,122.0,12752.1,32.0,...,1628.0,1983.2,399.4,121.0,6.0,NaN,975.1,gen_exist_pcap_z,2050,FX


In [19]:
# Combine/merge hydro and other techs

combined_exist_pcap_z = pd.concat(
    [
        excel_exist_pcap_z,
        exist_pcap_z,
    ]
).loc[:, ["Technology", "Year", "parameter", "limtype"] + zones]

combined_exist_pcap_r = pd.concat(
    [
        excel_exist_pcap_r,
        exist_pcap_r,
    ]
)

combined_exist_pcap_z = combined_exist_pcap_z.replace({np.nan:""})
combined_exist_pcap_r = combined_exist_pcap_r.replace({np.nan:""})

combined_exist_pcap_z

,Technology,Year,parameter,limtype,AL,AT,BA,BE,BG,CH,...,NO,PL,PT,RO,RS,SE,SI,SK,XK,UK
0,HydroRoR,2050,gen_exist_pcap_z,UP,370.4,4566.8,0.0,59.0,22.4,4008.4,...,0.0,14.4,1246.5,654.4,431.0,1995.0,394.0,646.5,0.0,848.5
1,HydroRes,2050,gen_exist_pcap_z,FX,1760.2,5079.9,0.0,12.7,1499.5,8922.3,...,31514.0,412.5,1368.4,6386.5,428.3,11667.6,602.2,804.4,0.0,524.0
2,HydroRes,2050,gen_exist_ecap_z,FX,3050385.2,3200000.0,2500000.0,22070.6,4000000.0,8400000.0,...,84349253.0,1600000.0,2600000.0,12100000.0,742233.8,33800000.0,2200000.0,2200000.0,0.0,908079.7
0,NuclearEPR,2050,gen_exist_pcap_z,FX,,,,,,,...,,,,,,,,471.0,,
1,Windoffshore,2050,gen_exist_pcap_z,FX,,,,706.0,,,...,108.0,,25.0,,,,,,,4674.5
2,Windonshore,2050,gen_exist_pcap_z,FX,,546.0,,417.0,33.6,14.0,...,1967.8,3225.0,266.6,108.0,,7447.4,,,,1168.1
3,Solar,2050,gen_exist_pcap_z,FX,,77.7,,39.0,1106.0,1.2,...,,1628.0,1983.2,399.4,,121.0,6.0,,,975.1


In [20]:
# Create csv files with the existing capacities by region (nuts2) and zone (country)
(combined_exist_pcap_z.to_csv(existing_pcap_z_path, index=False))

(combined_exist_pcap_r.to_csv(existing_pcap_r_path, index=False))

(excel_gen_lim_z.to_csv(gen_lim_z_path, index=False))


In [21]:
# Load data
existing_data = pyexcel_ods.get_data(ods_file)


updated_dataframes = [combined_exist_pcap_z, combined_exist_pcap_r]
sheets = ["gen_exist_z", "gen_exist_r"]

for updated_dataframe, sheet in zip(updated_dataframes,sheets):
    # Update sheet
    existing_data[sheet] = updated_dataframe.values.tolist() ## add values
    existing_data[sheet].insert(0, list(updated_dataframe.columns)) ## add headers
    
# Write again the file
pyexcel_ods.save_data(output_file_path, existing_data)

# Replace sheet from ods file with csv file

In [ ]:
import pyexcel_ods
import pandas as pd

def replace_sheet_with_csv(ods_file, output_file, csv_paths, sheet_names):
    '''
    Replace the sheets of an odf file with values contained in csv files
    ods_file: string with the odf path
    output_file: string with the output path to store the new ods file
    updated_dataframes: list of csv paths to replace
    sheet_names: list of sheet names to be replaced
    '''
    updated_dataframes = [pd.read_csv(csv_path) for csv_path in csv_paths]
    existing_data = pyexcel_ods.get_data(ods_file)

    for updated_dataframe, sheet_name in zip(updated_dataframes,sheet_names):
        
        existing_data[sheet_name] = updated_dataframe.values.tolist()
        existing_data[sheet_name].insert(0, list(updated_dataframe.columns))

    pyexcel_ods.save_data(output_file, existing_data)


In [ ]:
year = 2050

ods_file = "highres_technoeconomic_database.ods"
output_file = "highres_technoeconomic_database_updated.ods"

csv_paths = [f"existing_pcap_z_{year}.csv", f"existing_pcap_r_{year}.csv"]
# csv_paths = [f"existing_pcap_r_{year}.csv", f"existing_pcap_z_{year}.csv", f"gen_lim_z_{year}.csv"]
sheet_names = ["gen_exist_z", "gen_exist_r"]
# sheet_names = ["gen_exist_z", "gen_exist_r", "gen_lim_z"]

replace_sheet_with_csv(ods_file, output_file, csv_paths, sheet_names)

# Plots

In [ ]:
plants_in_europe_path = "plants_in_europe.png"
pcap_in_eur_path = "capacity_by_nuts2.png"

In [ ]:
### Merge all the plants for plotting
techs_plot = pd.concat(
    [
        no_error_techs,
        offshore_plants,
        country_border_plants,
    ]
).reset_index(drop=True)
techs_plot = gpd.GeoDataFrame(
    techs_plot,
    crs="epsg:4326",
)



In [ ]:
## plot of each plant

fig, ax = plt.subplots(figsize=(10, 10))

euroshape.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5)
techs_plot.query("Technology == 'solar'").plot(
    ax=ax, markersize=1, color="gold", label="Solar"
)
techs_plot.query("Technology == 'onshore'").plot(
    ax=ax, markersize=1, color="tab:green", label="Onshore"
)
techs_plot.query("Technology == 'offshore'").plot(
    ax=ax, markersize=1, color="tab:blue", label="Offshore"
)
techs_plot.query("Technology == 'nuclear'").plot(
    ax=ax, markersize=1, color="tab:red", label="Nuclear"
)
ax.legend()

fig.savefig(
    plants_in_europe_path, dpi=300, format="png", bbox_inches="tight"
)


In [ ]:
## For plotting
installed_cap = (
    techs_plot.assign(capacity_GW=techs_plot["capacity_MW"].div(1000))
    .loc[:, ["Technology", "index", "capacity_GW"]]
    .set_index("index")
)


installed_cap = installed_cap.merge(
    euroshape.loc[:, ["geometry"]],
    left_on="index",
    right_on="index",
    how="left",
)

installed_cap = gpd.GeoDataFrame(installed_cap)

onshore_cap = (
    installed_cap.query("Technology == 'onshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

offshore_cap = (
    installed_cap.query("Technology == 'offshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

solar_cap = (
    installed_cap.query("Technology == 'solar'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

nuclear_cap = (
    installed_cap.query("Technology == 'nuclear'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

In [ ]:
## plot capacities by region


fig, ax = plt.subplots(2, 2, figsize=(10, 12))

# onshore
cmap = plt.get_cmap("magma_r")
onshore_cap.plot(
    column="capacity_GW", ax=ax[0][0], markersize=0.5, cmap=cmap, legend=True
)
euroshape.plot(ax=ax[0][0], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][0].set_title("Onshore capacities (GW)", fontsize=12, weight="bold")

# offshore
cmap = plt.get_cmap("magma_r")
offshore_cap.plot(
    column="capacity_GW", ax=ax[0][1], markersize=0.5, cmap=cmap, legend=True
)
euroshape.plot(ax=ax[0][1], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][1].set_title("Offshore capacities (GW)", fontsize=12, weight="bold")

# solar
cmap = plt.get_cmap("magma_r")
solar_cap.plot(
    column="capacity_GW", ax=ax[1][0], markersize=0.5, cmap=cmap, legend=True
)
euroshape.plot(ax=ax[1][0], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][0].set_title("Solar capacities (GW)", fontsize=12, weight="bold")

# nuclear
cmap = plt.get_cmap("magma_r")
nuclear_cap.plot(
    column="capacity_GW", ax=ax[1][1], markersize=0.5, cmap=cmap, legend=True
)
euroshape.plot(ax=ax[1][1], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][1].set_title("Nuclear capacities (GW)", fontsize=12, weight="bold")

plt.tight_layout()

fig.savefig(pcap_in_eur_path, dpi=300, format="png", bbox_inches="tight")